# Exercise 01: Learning from Data – Loss, Train/Test, Cross-Validation, Bias–Variance

**Dataset:** Daily bike rentals of *Capital Bikeshare* in Washington D.C. (2011–2012), enriched with weather data
(Fanaee-T & Gama, 2013). Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)
/ [Kaggle](https://www.kaggle.com/datasets/lakshmi25npathi/bike-sharing-dataset).

**Business question:** How many bikes will be rented on a day with a given temperature? And – more importantly for this
exercise – *how complex* should a model be so that it also works on **new** days?

**Slides:** Part 02 «Learning from data» (loss functions, train/test split, k-fold cross-validation, under-/overfitting,
bias–variance trade-off).

**After this exercise you can ...**
- compute and compare squared and absolute loss by hand,
- explain why the training error is optimistically biased,
- use a hold-out split and k-fold cross-validation to choose model complexity,
- recognise under- and overfitting and relate them to bias and variance.

## Concept

**Loss and training objective.** A loss measures the error of *one* prediction. Training minimises the average loss
over the $n$ training examples:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2, \qquad
\text{RMSE} = \sqrt{\text{MSE}}, \qquad
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|$$

Squared loss punishes large errors heavily (outlier-sensitive); absolute loss grows linearly (more robust).

**Generalisation error.** What we care about is the error on **new, unseen** data. It cannot be computed directly, so we
estimate it on data that were *not* used for fitting:

| approach | idea |
|---|---|
| hold-out split | fit on the training set (e.g. 80 %), evaluate once on the test set (20 %) |
| k-fold cross-validation | split the training set into $k$ folds; fit on $k-1$ folds, validate on the remaining one; repeat $k$ times; CV error $=\frac{1}{k}\sum_j \text{MSE}_j$ |

The test set is touched **only once** at the very end – hyperparameters (e.g. the polynomial degree) are chosen with CV.

**Under- and overfitting / bias–variance.** For squared-error loss the expected test error decomposes into

$$E\big[(y-\hat f(x))^2\big] = \text{Bias}^2 + \text{Variance} + \sigma^2_{\text{noise}}$$

- *Underfitting* (high bias): model too simple → high training **and** test error.
- *Overfitting* (high variance): model too flexible → low training error, high test error; the fit changes a lot with
  another training sample.
- The goal is to minimise the **sum**, not either term alone.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Loss functions by hand

Two models predict the daily number of rentals for five days:

| day | observed $y$ | model A $\hat y$ | model B $\hat y$ |
|---|---|---|---|
| 1 | 1200 | 1500 | 1250 |
| 2 | 2500 | 2300 | 2550 |
| 3 | 3100 | 3300 | 3050 |
| 4 | 4000 | 3700 | 4050 |
| 5 | 4600 | 4500 | 3800 |

a) Compute the residuals, the **MSE**, the **RMSE** and the **MAE** for both models **by hand** (paper or calculator).

b) Which model is better according to RMSE, and which according to MAE? Explain the difference.

c) Verify your results with `numpy`.

In [ ]:
y       = np.array([1200, 2500, 3100, 4000, 4600])
y_hat_a = np.array([1500, 2300, 3300, 3700, 4500])
y_hat_b = np.array([1250, 2550, 3050, 4050, 3800])

for name, y_hat in [('A', y_hat_a), ('B', y_hat_b)]:
    res  = y - y_hat
    mse  = np.mean(res**2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(res))
    print(f'Model {name}: residuals = {res}, MSE = {mse:,.0f}, RMSE = {rmse:.1f}, MAE = {mae:.1f}')

**Solution:**

Model A: residuals −300, 200, −200, 300, 100 → MSE = 270'000/5 = 54'000, **RMSE ≈ 232.4**, **MAE = 220**.
Model B: residuals −50, −50, 50, −50, 800 → MSE = 650'000/5 = 130'000, **RMSE ≈ 360.6**, **MAE = 200**.

According to **RMSE model A** is better, according to **MAE model B** is better. Model B is almost perfect on four days
but makes one large error (800). Squared loss weighs this single error heavily (800² = 640'000), absolute loss only
linearly. Which loss to use depends on the costs of large errors: if one big mistake (e.g. far too few bikes available)
is very costly, squared loss is the better choice.

## Exercise 2: Import and explore the bike sharing data

The column `temp` contains the *normalised* temperature (°C divided by 41). `cnt` is the total number of rentals per day.

a) Import the data, convert `dteday` to a date and create a column `temp_c` with the temperature in °C.

b) To avoid mixing in the strong growth of the service between 2011 and 2012, keep **only the year 2012** (`yr == 1`).

c) Create a scatter plot of `cnt` against `temp_c`. Is the relationship linear?

In [ ]:
# Import the data
df_orig = pd.read_csv('./Data/bike_sharing_day.csv', sep=',')

# a) Convert date, create temp_c
df_orig['dteday'] = pd.to_datetime(df_orig['dteday'])
df_orig['temp_c'] = df_orig['temp'] * 41

# b) Keep only 2012
df = df_orig.loc[df_orig['yr'] == 1, ['dteday', 'temp_c', 'hum', 'windspeed', 'workingday', 'cnt']]
df = df.reset_index(drop=True)
print(df.shape)
print(df.head())

# c) Scatter plot cnt vs. temp_c
plt.figure(figsize=(7, 4))
plt.scatter(df['temp_c'], df['cnt'], s=12, alpha=0.6)
plt.xlabel('Temperature [°C]')
plt.ylabel('Rentals per day')
plt.title('Daily bike rentals vs. temperature (2012)')
plt.grid(alpha=0.3)
plt.show()

**Solution:**

The relationship is **not linear**: rentals increase with temperature up to about 25–28 °C and then flatten or even
decrease on very hot days. A straight line will therefore have a systematic error (bias); a curved model (e.g. a
polynomial) should fit better. At the same temperature the number of rentals varies a lot (rain, weekdays, holidays) –
this is noise that no model based on temperature alone can remove.

## Exercise 3: Hold-out split and polynomial models

We describe the number of rentals as a polynomial of the temperature

$$\hat y = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_d x^d .$$

This is still a *linear* regression (linear in the parameters $\beta$). The **degree $d$** controls the model complexity.
The helper function `make_poly_model(degree)` below builds such a model (the features are standardised for numerical
stability).

a) Split the data into a training set (80 %) and a test set (20 %) with `random_state=42`.

b) Fit a polynomial of degree 1 (= simple linear regression) on the training set and compute the RMSE on the training
and on the test set.

In [ ]:
def make_poly_model(degree):
    """Polynomial regression model of the given degree."""
    return make_pipeline(PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(),
                         LinearRegression())

In [ ]:
X = df[['temp_c']]
y = df['cnt']

# a) Train/test split (80/20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)

# b) Fit a polynomial of degree 1 and compute train and test RMSE
model_1 = make_poly_model(1).fit(X_train, y_train)
print('Train RMSE:', round(root_mean_squared_error(y_train, model_1.predict(X_train))))
print('Test RMSE: ', round(root_mean_squared_error(y_test,  model_1.predict(X_test))))

## Exercise 4: Under- and overfitting with a small training sample

Overfitting is most visible when there are only few training examples. Imagine the bike sharing company has only
**30 days** of data.

a) Draw a random sample of 30 days from the training set (`X_train.sample(30, random_state=19)`, same rows for `y_train`).

b) For each degree $d = 1, 2, \dots, 15$ fit `make_poly_model(d)` on the 30 days and compute the RMSE on these 30
training days and on the (full) test set. Plot both RMSE curves against the degree (use a logarithmic y-axis).

c) Plot the fitted curves for the degrees 1, 3 and 12 together with the 30 training points.

d) Which degree underfits, which overfits, which fits well? Why does the training error keep decreasing?

In [ ]:
# a) Sample of 30 training days
X_small = X_train.sample(30, random_state=19)
y_small = y_train.loc[X_small.index]

# b) Train and test RMSE for degree 1..15
degrees = range(1, 16)
rmse_train, rmse_test = [], []
for d in degrees:
    m = make_poly_model(d).fit(X_small, y_small)
    rmse_train.append(root_mean_squared_error(y_small, m.predict(X_small)))
    rmse_test.append(root_mean_squared_error(y_test, m.predict(X_test)))

print(pd.DataFrame({'degree': degrees,
                    'train RMSE': np.round(rmse_train),
                    'test RMSE': np.round(rmse_test)}).to_string(index=False))

plt.figure(figsize=(7, 4))
plt.plot(degrees, rmse_train, marker='o', label='training error (30 days)')
plt.plot(degrees, rmse_test, marker='o', label='test error')
plt.yscale('log')
plt.xlabel('Polynomial degree d')
plt.ylabel('RMSE (log scale)')
plt.title('Training vs. test error')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# c) Fitted curves for degree 1, 3 and 12
x_grid = pd.DataFrame({'temp_c': np.linspace(X_small['temp_c'].min(), X_small['temp_c'].max(), 300)})

plt.figure(figsize=(7, 4))
plt.scatter(X_small['temp_c'], y_small, color='black', s=20, label='30 training days')
for d in [1, 3, 12]:
    m = make_poly_model(d).fit(X_small, y_small)
    plt.plot(x_grid['temp_c'], m.predict(x_grid), linewidth=2, label=f'degree {d}')
plt.ylim(0, 10000)
plt.xlabel('Temperature [°C]')
plt.ylabel('Rentals per day')
plt.title('Polynomial fits on 30 training days')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

- **Degree 1 underfits** slightly: the straight line cannot follow the flattening at high temperatures (bias).
- **Degree 2–3 fit well**: they capture the curvature and have the lowest test error.
- **High degrees (≥ 4) overfit**: the curve wiggles through the 30 training points, the training error is lowest, but
  the test error grows strongly (by more than a factor of 10 for degree ≥ 8 – the curve shoots up or down between and
  beyond the training points).

The training error can only decrease with the degree, because a polynomial of degree $d+1$ contains the polynomial of
degree $d$ as a special case ($\beta_{d+1}=0$). (The tiny increases you may see in the table are numerical rounding
effects of the badly conditioned high-degree polynomials.) The training error is therefore an *optimistically biased*
estimate of the error on new data.

## Exercise 5: Choose the degree with 5-fold cross-validation

In practice we must not look at the test set to choose the degree. Instead we use k-fold cross-validation **on the
training data only**.

a) For $d = 1, \dots, 10$ compute the 5-fold CV RMSE on the **30 training days**
(use `cross_val_score(..., scoring='neg_root_mean_squared_error', cv=KFold(5, shuffle=True, random_state=42))`;
note the minus sign). Which degree does CV select?

b) Repeat a) with the **full training set** (all ≈ 290 days). Which degree is selected now and how large is the
difference between degree 3 and degree 10?

c) Fit the selected model on the full training set and evaluate it **once** on the test set.

d) Why does overfitting become much less severe with more data?

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_rmse(degree, X, y):
    """Mean 5-fold CV RMSE of a polynomial model."""
    scores = cross_val_score(make_poly_model(degree), X, y, cv=cv,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

degrees_cv = range(1, 11)
cv_small = [cv_rmse(d, X_small, y_small) for d in degrees_cv]
cv_full  = [cv_rmse(d, X_train, y_train) for d in degrees_cv]

print(pd.DataFrame({'degree': degrees_cv,
                    'CV RMSE (30 days)': np.round(cv_small),
                    'CV RMSE (full training set)': np.round(cv_full)}).to_string(index=False))
print('Selected degree (30 days):          ', degrees_cv[int(np.argmin(cv_small))])
print('Selected degree (full training set):', degrees_cv[int(np.argmin(cv_full))])

plt.figure(figsize=(7, 4))
plt.plot(degrees_cv, cv_small, marker='o', label='30 training days')
plt.plot(degrees_cv, cv_full, marker='o', label=f'{len(X_train)} training days')
plt.yscale('log')
plt.xlabel('Polynomial degree d')
plt.ylabel('5-fold CV RMSE (log scale)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# c) Fit the selected model on the full training set, evaluate once on the test set
best_degree = degrees_cv[int(np.argmin(cv_full))]
final_model = make_poly_model(best_degree).fit(X_train, y_train)
print(f'Degree {best_degree}: test RMSE =',
      round(root_mean_squared_error(y_test, final_model.predict(X_test))))

**Solution:**

With only 30 days, CV selects a low degree (2) and the CV error increases strongly for high degrees – just like the test error in
Exercise 4, but **without** using the test set. With the full training set the CV curve is almost flat from degree 3 on:
the selected degree has a CV RMSE of roughly 1'100 and degree 10 is only slightly worse. With many observations, the
extra parameters of a flexible model are pinned down by the data – the **variance** of the fit shrinks. Overfitting is
always a question of model complexity *relative to the amount of data*.

The remaining error of about 1'000 rentals per day is mostly **noise from the perspective of this model** (weather,
weekday, holidays); it can only be reduced by adding further features, not by a more complex function of temperature.

## Exercise 6: Visualise bias and variance

Bias and variance describe what happens if we **repeat** the whole training on *other* samples of the same size.

a) Draw 50 different random samples of 30 days from the training set (`random_state=0, 1, ..., 49`). For each sample
fit a polynomial of degree 1, 3 and 12. Plot the 50 fitted curves per degree in three subplots
(use `plt.ylim(0, 10000)`).

b) For each degree, compute the mean and the standard deviation of the 50 predictions at **25 °C**.
What do you observe? Relate your observations to bias and variance.

In [ ]:
x_grid = pd.DataFrame({'temp_c': np.linspace(5, 33, 200)})
x_25   = pd.DataFrame({'temp_c': [25.0]})

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
pred_25 = {1: [], 3: [], 12: []}

for ax, d in zip(axes, [1, 3, 12]):
    for seed in range(50):
        X_s = X_train.sample(30, random_state=seed)
        y_s = y_train.loc[X_s.index]
        m = make_poly_model(d).fit(X_s, y_s)
        ax.plot(x_grid['temp_c'], m.predict(x_grid), color='steelblue', alpha=0.25, linewidth=1)
        pred_25[d].append(m.predict(x_25)[0])
    ax.scatter(X_train['temp_c'], y_train, color='black', s=4, alpha=0.3)
    ax.set_title(f'degree {d}: 50 fits on 30 random days')
    ax.set_xlabel('Temperature [°C]')
    ax.set_ylim(0, 10000)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Rentals per day')
plt.show()

# b) Mean and standard deviation of the predictions at 25 °C
print(pd.DataFrame({d: [np.mean(p), np.std(p)] for d, p in pred_25.items()},
                   index=['mean prediction at 25 °C', 'std of predictions at 25 °C']).round(0))

**Solution:**

- **Degree 1:** the 50 lines are similar to each other (lowest standard deviation at 25 °C), but they all miss the
  curvature in the same way: they systematically underestimate rentals around 25 °C (mean prediction clearly below the
  one of degree 3) and overestimate them on very hot days (high **bias**).
- **Degree 3:** the curves follow the shape of the data and still vary only moderately – a good compromise.
- **Degree 12:** the curves differ wildly from sample to sample, especially at the edges of the temperature range
  (high **variance**). Even at 25 °C, inside the range of the data, the standard deviation of the predictions is clearly
  larger than for degree 3.

This is exactly the bias–variance trade-off: a simple model is *stable but systematically wrong*, a very flexible model
is *right on average but unstable*. The best test error is achieved in between.

## Exercise 7: Reflection

a) A colleague chooses the polynomial degree by trying all degrees and picking the one with the smallest **test** error.
What is the problem?

b) Why is k-fold cross-validation usually preferred over a single validation split when the data set is small?

c) Name two ways to reduce the variance of a model and one way to reduce its bias.

**Solution:**

a) The test set is then used for model selection; the reported test error is the *minimum* over many models and
therefore optimistically biased. The test set no longer measures performance on truly unseen data. Hyperparameters must
be chosen with a validation set or cross-validation on the training data.

b) With a single split, the result depends strongly on which few observations end up in the validation set. k-fold CV
uses every observation once for validation and averages over $k$ estimates, which gives a more stable estimate and uses
the data more efficiently.

c) Lower variance: collect more training data, use a simpler model (lower degree), regularisation/pruning, or averaging
many models (bagging, random forest). Lower bias: use a more flexible model (e.g. higher degree, trees) or add
informative features (e.g. humidity, working day).

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')